In [ ]:
import os
import pandas as pd

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print("Working folder:", os.getcwd())

df = pd.read_csv('data/processed/movies_features.csv', parse_dates=['release_date'])
print(df.shape)


Working folder: C:\Users\ycode\Desktop\Projet\Machine Learning\CineMatch AI
(2972, 37)


In [ ]:
threshold = df["vote_count"].quantile(0.75)
print(threshold)
df["high_engagement"] = (df["vote_count"] >= threshold).astype(int)

df[["movie_id", "title", "vote_count", "high_engagement"]].head()


5290.0


,movie_id,title,vote_count,high_engagement
0,969681,Spider-Man: Brand New Day,3022,0
1,1599191,Renegade Immortal: Battle of the Immortal Slayer,12,0
2,1423191,Resident Evil,750,0
3,1368337,The Odyssey,4016,0
4,1228834,The Fix,46,0


In [ ]:
from sklearn.model_selection import train_test_split

numeric_features = ["runtime", "movie_age", "release_year", "n_genres", "n_keywords", "overview_word_count"]
categorical_features = ["original_language", "runtime_category", "main_genre"]
genre_cols = [col for col in df.columns if col.startswith("genre_")]

X = df[numeric_features + categorical_features + genre_cols]
y = df["high_engagement"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape, y_train.value_counts().to_dict()


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
    ("genre", "passthrough", genre_cols),
])

preprocessor


In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

log_reg_pipeline.fit(X_train, y_train)
print("Trained logistic regression")


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred = log_reg_pipeline.predict(X_test)
y_proba = log_reg_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")),
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_rf))


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

svm_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=5000)
    )),
])

svm_pipeline.fit(X_train, y_train)

y_pred_svm = svm_pipeline.predict(X_test)
y_proba_svm = svm_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_svm))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_svm))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_svm))


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    rf_pipeline, X, y, cv=skf,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"]
)

for metric in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]:
    print(metric, "mean:", round(scores[metric].mean(), 3), "std:", round(scores[metric].std(), 3))


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 4],
}

grid_search = GridSearchCV(
    rf_pipeline, param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="f1",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best F1 (cross-val):", grid_search.best_score_)


In [ ]:
best_rf_pipeline = grid_search.best_estimator_

y_pred_tuned = best_rf_pipeline.predict(X_test)
y_proba_tuned = best_rf_pipeline.predict_proba(X_test)[:, 1]

print("--- Default Random Forest (before) ---")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

print()
print("--- Tuned Random Forest (after) ---")
print(classification_report(y_test, y_pred_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_tuned))


In [ ]:
import joblib

os.makedirs("models", exist_ok=True)
joblib.dump(best_rf_pipeline, "models/classifier_rf.pkl")
print("saved")
